# Hot-Path Memory with LangMem

**Hot-path memory** means the agent manages its own memories *during* inference — on the same path as the LLM call itself.

This notebook uses `create_manage_memory_tool` from LangMem, which gives the agent a callable tool to **create, update, and delete** entries in a shared `InMemoryStore`. The agent decides what to remember and when.

**Contrast with background (off-path) memory**, where a separate process extracts memories *after* the conversation ends — the agent doesn't drive it directly.

### Memory lifecycle in this pattern
1. **Retrieval** — the `prompt` function searches the store and injects relevant memories into the system prompt before each LLM call
2. **Storage** — when the user says "remember that", the agent calls `manage_memory_tool` to write to the store
3. **Persistence across threads** — stored memories are available in any future conversation, regardless of thread

---

## Imports

In [ ]:
# LangGraph primitives
from langgraph.checkpoint.memory import MemorySaver      # Persists conversation history (short-term, per-thread)
from langgraph.store.memory import InMemoryStore         # In-process vector store for long-term memories

# create_agent replaces the deprecated create_react_agent (moved in LangGraph v1.0)
from langchain.agents import create_agent                # Builds a ReAct-style agent graph

# dynamic_prompt: decorator that creates middleware for injecting a dynamic system prompt
# ModelRequest: carries the current state + runtime (store, context) into the middleware
from langchain.agents.middleware import dynamic_prompt
from langchain.agents.middleware.types import ModelRequest

# LangMem tool
from langmem import create_manage_memory_tool, create_search_memory_tool            # Gives the agent a tool to create/update/delete memories

## The Memory Prompt (Middleware)

Instead of a raw `prompt` function, `create_agent` uses the `@dynamic_prompt` decorator to inject a dynamic system prompt before each model call.

The decorated function receives a `ModelRequest` which contains:
- `request.state["messages"]` — the current conversation history
- `request.runtime.store` — the `InMemoryStore` injected by LangGraph (no need for `get_store()`)

This is the **retrieval** step: we search for memories relevant to the latest user message and inject them into the system prompt.

In [3]:
@dynamic_prompt
def memory_prompt(request: ModelRequest) -> str:
    """
    Middleware that injects relevant memories into the system prompt before each LLM call.

    @dynamic_prompt turns this function into an AgentMiddleware that intercepts the model
    call, modifies the system prompt, and forwards to the actual model handler.
    """
    # Access the store via runtime — LangGraph injects it automatically.
    # This replaces the old get_store() pattern used with create_react_agent.
    store = request.runtime.store

    # Extract the latest human message to use as the memory search query.
    messages = request.state["messages"]
    user_messages = [m for m in messages if getattr(m, "type", None) == "human"]
    query = user_messages[-1].content if user_messages else ""

    # Semantic search: find stored memories most relevant to the current user message.
    # The namespace ("memories",) scopes the search to our memory partition.
    # Tip: use ("memories", user_id) to isolate each user's memories in multi-user apps.
    memories = store.search(("memories",), query=query)

    return f"""You are a helpful assistant.

## Memories
<memories>
{memories}
</memories>
"""

## Setting Up the Agent

Three components work together:

| Component | Role |
|---|---|
| `InMemoryStore` | **Long-term** memory store — persists across threads. Supports vector search for semantic recall. |
| `MemorySaver` | **Short-term** checkpointer — saves the conversation history (messages) per thread. |
| `create_agent` | Builds the agent graph, wiring middleware, tools, store, and checkpointer together. |

**Key distinction:**
- `InMemoryStore` → long-term, **cross-thread** (memories the agent explicitly saves using its tool)
- `MemorySaver` → short-term, **per-thread** (the running conversation history within a session)

**Old vs new API:**

| `create_react_agent` (deprecated) | `create_agent` (current) |
|---|---|
| `prompt=<callable>` | `middleware=[memory_prompt]` via `@dynamic_prompt` |
| `get_store()` inside prompt fn | `request.runtime.store` inside middleware |

In [4]:
# Long-term memory store with vector search support.
# "dims": 768 matches the output dimension of the embedding model used below.
# Any memory the agent explicitly saves via manage_memory_tool will be stored here.
store = InMemoryStore(
    index={
        "dims": 768,                                       # Must match the embedding model's output size
        "embed": "google_genai:gemini-embedding-001",      # Embeds memories so they can be searched semantically
    }
)

# Short-term checkpointer: saves the graph state (message history) after each node.
# This lets you resume or continue a conversation by passing the same thread_id in config.
checkpointer = MemorySaver()

# Build the agent using the new create_agent API.
agent = create_agent(
    "google_genai:gemini-2.5-flash",
    middleware=[memory_prompt],   # Runs before each model call; injects memories into the system prompt
    tools=[
        # This tool gives the agent explicit control over its own memory.
        # It can call manage_memory to create, update, or delete entries in the store.
        # Namespace scopes where memories live — use ("memories", "{user_id}") for per-user isolation.
        create_manage_memory_tool(namespace=("memories",)),
    ],
    store=store,                  # Long-term store; also available as request.runtime.store in middleware
    checkpointer=checkpointer,    # Short-term checkpointer scoped per thread_id
)

## Demo: Memory Persists Across Conversations

The three steps below show the full memory lifecycle:

1. **No memory yet** — agent can't answer a preference question
2. **Save a memory** — agent stores the preference via `manage_memory_tool`
3. **New thread, recalled memory** — a fresh conversation thread retrieves the saved preference

> **Thread vs Memory:** A *thread* (`thread_id`) represents one conversation session — its history lives in the checkpointer (`MemorySaver`). *Memories* live in the `InMemoryStore` and are shared across all threads.

In [5]:
# Each conversation is identified by a thread_id.
# The checkpointer uses this to save and restore message history between invocations.
config = {"configurable": {"thread_id": "thread-a"}}

# --- Step 1: Ask before any memories have been saved ---
# The store is empty, so the prompt function finds nothing and the agent has no context.
response = agent.invoke(
    {"messages": [{"role": "user", "content": "Know which display mode I prefer?"}]},
    config=config,
)
print("Response (no memory yet):")
print(response["messages"][-1].content)
# Expected: The agent says it doesn't know — the store is empty.

Response (no memory yet):
[{'type': 'text', 'text': "I don't know yet, but I can remember it if you tell me. ", 'extras': {'signature': 'CtIFAb4+9vtwCWd4CiESHSp/H6HKqqhcQ4wOrSHYpDTV0Xkt7OespOC89jKFPcceIHyfdfbrHviAVbdKE+PQDpR/YAhr3ULXOuQkp0eSMhwy/0VKPP72Mv6HSM8Uo3Y+lhzIKFEz/7px9qNjBGRDbTI7NROMzhz4ZBh2oHNzsFkoW0wU0MgVlp2T1bZY3lWuueyFy3I4xT8HNWqJWmyA/1Et8aw/6Ul/YcAwP0INaxcDZWSRXrEabqmt8yPch9xHsXoMiffjoK6g6Gj4eqvEQJcR7bQLkUIHtPAQ/cxvqWsabOfDE5Pz6JS6iH292p9UL5kzB9KkhSyQKG7wjyXASVQxM5aFF+WdbiCi23duykZOIi3XclLPiYCxvDjM9fDeRfMOZdCwp26tLcdShZYDMTNyrrfwtqYJ7S897U7mQCsuUHWW0BN/CdRfjXpAnhNoJi+9t4jX4+8gSQbKh0vq3ZnH3p817JiQrZIzJ4UVRpHyziieMIciN4iQs09CxLfJ9p6EqP7yZrK+S6ajGcufoSqnMQxHiAau+tMByuC3vYCtXOCgAtmyVHnawVyD9cVvTyQou4M4ZzqfJ2U+YSSMvLu15ukQBZBttoK3kR2WMg6vdY2fR90pCh6onp9Bfi2EG53K8OVIE00Xp7BxAr7y1PeAU/J4LOriZ4i9QlC2umBIbYaJRb2mIcjVQpQyksLvW9SSRPgOKU5L1ri5F5oBLMZUM0vY2Uuz0vH+bliPBFzF8iHVTiT9WJVQtR6ubOZTw+IQR9PtSNdGnQkfnxZxv5ELEbIc52JaNS/QneESqoqZJdWsrTOFDIaZkKys2uPGgc/IET9V9Ojx7VhiRsRUPaPPUuTvjeA

In [6]:
# --- Step 2: Tell the agent your preference ---
# The agent will call manage_memory_tool to persist this to the InMemoryStore.
# We reuse config (thread-a) to continue the same conversation thread.
agent.invoke(
    {"messages": [{"role": "user", "content": "dark. Remember that."}]},
    config=config,
)
print("Preference saved. The agent stored it in the InMemoryStore.")

Preference saved. The agent stored it in the InMemoryStore.


In [7]:
# --- Step 3: New thread — but the memory still exists ---
# A new thread_id means a fresh conversation with no shared message history.
# However, the InMemoryStore is shared across ALL threads, so saved memories carry over.
new_config = {"configurable": {"thread_id": "thread-b"}}

response = agent.invoke(
    {"messages": [{"role": "user", "content": "Hey there. Do you remember me? What are my preferences?"}]},
    config=new_config,
)
print("Response (new thread, memory recalled):")
print(response["messages"][-1].content)
# Expected: The agent recalls the dark mode preference from the store,
# even though this is a brand-new conversation thread.

Response (new thread, memory recalled):
[{'type': 'text', 'text': "I remember that your display mode preference is dark.\n\nIs there anything else you'd like me to remember or any new preferences you'd like to set?", 'extras': {'signature': 'Cr0KAb4+9vu/MHcauGrUqSY3cp6Ozen6GYniylf9RnluUN6feIVd26zl6SEItykuMJQveI0TgjPCMzgEa+mdwwpz7bDHIlX2vbZiL/hgAG9Gc0DhDpHLjxJ943JBnqlMbPWHYclZ7/6ojfHMFp04Njxy0JrVUpVi6sWzwaladDjHhL7Qdt3O+lkq+8ByUv+wkNkwWvOSlBRVBKAnLvmj4B83dN7W+1t1q0iWijpbvnKmC+9W7yaVpt/vfNuCdKRATYjY9Vv5bkp0GunepYrJ73pq5VqZT26IQ1okg4Nl5w9alKXLQEhwvlhTe0LhOmnst56VhKo/xoD7gr25OsPzZ3uWMYKHTRtmw40GxN7icaTb1eF0c11pv+fC24AWzRSpnH+dVBrdXrgOBA9zeP25GxKfPjcu9oVi6xljjzu0tu6MQHmDtBWFW7sPIwid3ufUHPY1Xq+kmBfP7k3CHX0DPNHx86ORGj+Aykt4NkvlWsnz5IVpk17LWhN/d8majUN2Zt3ergirVG2c6pD3RLdRokIKfg+xpR9WfibiFt0K/LfrpS8oBpX+DldkPINtIE+G0MTyPChPCdDSunSTv26FkyMncP3q0j0o0WRm/8CDoPCwDJG6B8SsrVPHZh+7KiMAyrkqAPyae2eMCKNO9Utam+02b+N6gWhQEHgyj1yAGhGZAh6kpZIUnZJukQ2nXRNx4A7LF8pu+n31ORZn5nMG5E0aVYZ/e0S2KysieEOJhuFpCmtLZ+futY